# 03 — Impact analysis

Impact answers: *where does this column come from?* (upstream) and *what breaks if I change it?* (downstream).

**Two intentional selections** for `lineage-demo`:

| Selection | Role | Direction | Expected |
|-----------|------|-----------|----------|
| `customers_report.customer_lifetime_value` | sink / report column | upstream | 3 columns |
| `orders_base.amount` | mid-pipeline column | downstream | 2 columns |

Downstream from the sink column is **empty** — that is correct, not an error.

In [ ]:
import importlib.util
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path
from types import ModuleType

_COLD_START_GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/ClearMetric-Labs/ClearMetric-Core/main'

_CACHED_NOTEBOOKS_DIR = Path('/Users/jonkim/.cache/clearmetric/github-main/examples/notebooks')

def _github_raw_base() -> str:
    paths = sys.modules.get("_paths")
    default = (
        paths.GITHUB_RAW_BASE if paths is not None else _COLD_START_GITHUB_RAW_BASE
    )
    return os.environ.get("CM_CLEARMETRIC_GITHUB_RAW_BASE", default)
def _fetch_repo_file(repo_relative: str, dest: Path) -> None:
    if dest.is_file():
        return
    paths = sys.modules.get("_paths")
    if paths is not None:
        paths._fetch_github_file(repo_relative, dest)
        return
    url = f"{_github_raw_base()}/{repo_relative}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            dest.write_bytes(response.read())
    except urllib.error.URLError as exc:
        raise FileNotFoundError(
            f"Failed to download {url}. Check network access and branch path."
        ) from exc
def _find_local_helpers(start: Path | None = None) -> Path | None:
    start = start or Path.cwd()
    for root in (start, *start.parents):
        nested = root / "examples" / "notebooks"
        if (nested / "_paths.py").is_file():
            return nested
        if (root / "_paths.py").is_file() and (root / "_notebook_setup.py").is_file():
            return root
    return None
def _resolve_setup_file(start: Path | None = None) -> Path:
    local = _find_local_helpers(start)
    if local is not None:
        return local / "_notebook_setup.py"
    paths = sys.modules.get("_paths")
    cache_dir = (
        paths.CACHED_NOTEBOOKS_DIR if paths is not None else _CACHED_NOTEBOOKS_DIR
    )
    setup_file = cache_dir / "_notebook_setup.py"
    if not setup_file.is_file():
        _fetch_repo_file("examples/notebooks/_notebook_setup.py", setup_file)
    return setup_file
def _exec_setup_module(setup_file: Path) -> ModuleType:
    spec = importlib.util.spec_from_file_location("_notebook_setup", setup_file)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {setup_file}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
_exec_setup_module(_resolve_setup_file()).setup_notebook()


In [ ]:
import json

from _paths import lineage_demo_project

PROJECT_DIR = lineage_demo_project()


## Mid-pipeline: downstream from `orders_base.amount`

In [ ]:
from clearmetric.compiler.impact import impact

DOWNSTREAM_AMOUNT = {
    "column:customer_totals.total_amount",
    "column:customers_report.customer_lifetime_value",
}
_, amount_down = impact(PROJECT_DIR, selection="orders_base.amount", direction="downstream")
assert set(amount_down.related_ids) == DOWNSTREAM_AMOUNT
print("orders_base.amount downstream:", sorted(amount_down.related_ids))

## Sink column: upstream from `customers_report.customer_lifetime_value`

In [ ]:
UPSTREAM_CLV = {
    "column:customer_totals.total_amount",
    "column:orders_base.amount",
    "column:raw_orders.amount",
}
compiled, upstream = impact(
    PROJECT_DIR,
    selection="customers_report.customer_lifetime_value",
    direction="upstream",
)
_, downstream_clv = impact(
    PROJECT_DIR,
    selection="customers_report.customer_lifetime_value",
    direction="downstream",
)
assert set(upstream.related_ids) == UPSTREAM_CLV
assert downstream_clv.related_ids == []
print("upstream:", sorted(upstream.related_ids))
print("downstream (empty for sink):", downstream_clv.related_ids)

## `emit_impact` — JSON output

In [ ]:
from clearmetric.core.validate import validate_impact_output_dict
from clearmetric.emitters.impact import emit_impact

impact_json = emit_impact(compiled, upstream, format="json", direction="upstream")
payload = json.loads(impact_json)
validate_impact_output_dict(payload)
print("keys:", sorted(payload.keys()))
print("derivation entries:", len(payload["derivation"]))
print(json.dumps(payload["derivation"], indent=2))

## Renderers — tree and mermaid

In [ ]:
print(emit_impact(compiled, upstream, format="tree", direction="upstream"))
print("\n--- mermaid ---")
print(emit_impact(compiled, upstream, format="mermaid", direction="upstream"))